# 05 — CSCA: Contrastive Structure-Property Alignment (Novel Algorithm)

**No API key required.** Train a contrastive model that aligns molecular fingerprints with physicochemical properties in a shared latent space.

This is the novel scientific contribution:
- InfoNCE loss (CLIP-style) for structure <-> property alignment
- Analogous to MoleculeSTM (Nature MI 2023) but aligns structure <-> property
- After training: zero-shot property-to-structure retrieval

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from chemvision.core.reproducibility import set_global_seed
set_global_seed(42)
print('Seeded all RNGs for reproducibility')

## 5.1 — Build training dataset

In [ ]:
from chemvision.data.dataset_builder import MolecularDatasetBuilder
from pathlib import Path

builder = MolecularDatasetBuilder(seed=42)
n_seeds = builder.add_seeds()
n_random = builder.add_random_molecules(n=100)
print(f'Seeds: {n_seeds}, Random: {n_random}, Total: {n_seeds + n_random}')

out_dir = Path('/tmp/csca_dataset')
stats = builder.build(out_dir)
print(f'\nDataset built: {stats.n_total} molecules')
print(f'  Train: {stats.n_train}  Val: {stats.n_val}  Test: {stats.n_test}')
print(f'  FP dim: {stats.fp_dim}  Prop dim: {stats.prop_dim}')
print(f'  Properties: {stats.prop_names}')
print(f'  Mean MW: {stats.mean_mw:.1f} +/- {stats.std_mw:.1f}')
print(f'  Lipinski pass rate: {stats.lipinski_pass_rate:.0%}')

In [ ]:
fps, props, splits = MolecularDatasetBuilder.load_arrays(out_dir)
print(f'Fingerprints: {fps.shape}')
print(f'Properties: {props.shape}')
print(f'Property columns: {stats.prop_names}')

## 5.2 — Train CSCA model

In [ ]:
from chemvision.models.csca import CSCATrainer, CSCAConfig

config = CSCAConfig(
    fp_dim=fps.shape[1],       # 2048
    prop_dim=props.shape[1],   # 8
    latent_dim=64,
    hidden_dim=128,
    temperature=0.07,
    learning_rate=1e-3,
    seed=42,
)

trainer = CSCATrainer(config)
result = trainer.train(fps, props, epochs=100, batch_size=32, patience=20, verbose=True)

print(f'\nTraining complete:')
print(f'  Epochs: {result.epochs}')
print(f'  Final loss: {result.final_loss:.4f}')
print(f'  Converged: {result.converged}')
print(f'  Val retrieval accuracy: {result.val_retrieval_acc:.1%}')

In [ ]:
# Plot training loss curve
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(result.loss_history, color='steelblue', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('InfoNCE Loss')
ax.set_title(f'CSCA Training — {result.epochs} epochs, final loss = {result.final_loss:.4f}')
ax.grid(True, alpha=0.3)
ax.axhline(result.final_loss, color='red', linestyle='--', alpha=0.5, label=f'Final: {result.final_loss:.4f}')
ax.legend()
plt.tight_layout()
plt.show()

## 5.3 — Zero-shot retrieval: given properties, find the molecule

In [ ]:
import pandas as pd

# Use the first 5 molecules as queries
query_props = props[:5]
retrieved = trainer.retrieve(query_props, fps, k=3)

# Load SMILES from the train parquet to show names
train_df = pd.read_parquet(out_dir / 'train.parquet')
all_smiles = train_df['smiles'].tolist() if 'smiles' in train_df else []

print('Zero-shot retrieval results:')
print('=' * 60)
for i, indices in enumerate(retrieved):
    print(f'\nQuery {i} (props: MW={props[i,0]:.0f}, LogP={props[i,1]:.1f}, QED={props[i,7]:.2f})')
    for rank, idx in enumerate(indices):
        print(f'  Top-{rank+1}: index={idx}, MW={props[idx,0]:.0f}, LogP={props[idx,1]:.1f}')

## 5.4 — Visualise the learned latent space

In [ ]:
# Encode all molecules to latent space
latent = trainer.encode(fps)  # (N, latent_dim)

# Simple 2D projection via PCA
centered = latent - latent.mean(axis=0)
U, S, Vt = np.linalg.svd(centered, full_matrices=False)
pca_2d = centered @ Vt[:2].T

# Colour by MW
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sc1 = axes[0].scatter(pca_2d[:, 0], pca_2d[:, 1], c=props[:, 0], cmap='viridis', s=30, alpha=0.7)
fig.colorbar(sc1, ax=axes[0], label='MW (g/mol)')
axes[0].set_title('CSCA latent space — coloured by MW')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[0].grid(True, alpha=0.2)

sc2 = axes[1].scatter(pca_2d[:, 0], pca_2d[:, 1], c=props[:, 7], cmap='RdYlGn', s=30, alpha=0.7)
fig.colorbar(sc2, ax=axes[1], label='QED')
axes[1].set_title('CSCA latent space — coloured by QED')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()